# EE2211 Tutorial 8 — Gradient Descent

Core identity used throughout: for $\mathbf{x}^{\top}\mathbf{w}$,
$$\nabla_{\mathbf{w}}(\mathbf{x}^{\top}\mathbf{w}) \;=\; \mathbf{x}.$$
Chain rule + this identity is all we need for Q2–Q5.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Question 1 — one step of GD on $f(x)=x^4$

Given $f(x)=x^4$, $x_0=2$, learning rate $\eta=0.1$.

$$f'(x)=4x^3 \;\Rightarrow\; f'(2)=4(2)^3=32$$
$$x_1 = x_0 - \eta\, f'(x_0) = 2 - 0.1 \cdot 32 = 2 - 3.2 = \boxed{-1.2}$$

Note the overshoot past the minimum $x^\star=0$ — $\eta=0.1$ is too large for this curvature, GD will oscillate and eventually diverge.

In [ ]:
# Q1: single GD step on f(x) = x^4
def f(x):
    return x ** 4

def grad_f(x):
    return 4 * x ** 3

x, lr = 2.0, 0.1
x_new = x - lr * grad_f(x)
print(f"x0 = {x}, grad at x0 = {grad_f(x)}, x1 = {x_new}")

# Sanity: iterate a few more steps -> oscillation / divergence
xs = [x]
for _ in range(6):
    xs.append(xs[-1] - lr * grad_f(xs[-1]))
print("first 7 iterates:", [round(v, 4) for v in xs])

## Question 2 — exponential model via gradient descent

Model: $f(\mathbf{x},\mathbf{w}) = \exp(-\mathbf{x}^{\top}\mathbf{w})$ with square loss $C(\mathbf{w}) = \sum_{i=1}^{m}(f(\mathbf{x}_i,\mathbf{w}) - y_i)^{2}$.

Per the handout, the gradient works out to

$$\nabla_{\mathbf{w}} C = \sum_{i=1}^{m} 2(f(\mathbf{x}_i,\mathbf{w}) - y_i)\,\nabla_{\mathbf{w}} f$$

with $\nabla_{\mathbf{w}} f = -\exp(-\mathbf{x}_i^{\top}\mathbf{w})\,\mathbf{x}_i = -f(\mathbf{x}_i,\mathbf{w})\,\mathbf{x}_i$, giving

$$\boxed{\;\nabla_{\mathbf{w}} C = -\sum_{i=1}^{m} 2\,(f(\mathbf{x}_i,\mathbf{w}) - y_i)\,f(\mathbf{x}_i,\mathbf{w})\,\mathbf{x}_i\;}$$

**Preprocessing**: divide `year` and `expenditure` by their respective max values so both live in $(0,1]$; augment $\mathbf{x}=[1,\text{year}]^{\top}$ for the bias. The PDF says to divide by 2018, but the supplied CSV now extends to 2022 — we divide by `max(years)` since only the *scaling* matters.

In [ ]:
# ---- forward + cost + gradient in one vectorized shot ----
# X: (m, d), w: (d,), y: (m,)  -> pred_y, cost (SSE), gradient (d,)
def exp_cost_gradient(X, w, y):
    pred_y = np.exp(-X @ w)                     # f(x_i, w) stacked
    cost   = np.sum((pred_y - y) ** 2)          # sum-squared error
    grad   = -2 * (pred_y - y) * pred_y @ X     # shape (d,), see derivation
    return pred_y, cost, grad

# ---- one GD training run, returns everything we need for the plots ----
DATA_PATH = "../data/Tutorial 2 supp government-expenditure-on-education.csv"

def run_gd(lr, num_iters=200_000, data_path=DATA_PATH):
    df          = pd.read_csv(data_path)
    expenditure = df["total_expenditure_on_education"].to_numpy().astype(float)
    years       = df["year"].to_numpy().astype(float)

    max_exp, max_year = expenditure.max(), years.max()
    y = expenditure / max_exp
    X = np.ones((len(y), 2))
    X[:, 1] = years / max_year

    w = np.zeros(2)
    cost_vec = np.zeros(num_iters)
    _, init_cost, _ = exp_cost_gradient(X, w, y)

    for i in range(num_iters):
        _, cost, grad = exp_cost_gradient(X, w, y)
        cost_vec[i] = cost
        w = w - lr * grad                       # <-- the GD update

    _, final_cost, _ = exp_cost_gradient(X, w, y)
    return dict(w=w, years=years, expenditure=expenditure,
                X=X, y=y, max_year=max_year, max_exp=max_exp,
                cost_vec=cost_vec, init_cost=init_cost, final_cost=final_cost,
                lr=lr, num_iters=num_iters)

# The PDF specifies num_iters = 2_000_000 — feel free to pass that in if you want
# to reproduce the reference final cost exactly; 200k is enough to see convergence.

### Q2 (a) — cost curve at $\eta = 0.03$

In [ ]:
res = run_gd(lr=0.03, num_iters=200_000)
print(f"Initial Cost = {res['init_cost']:.6f}")
print(f"Final   Cost = {res['final_cost']:.6f}")
print(f"Learned w    = {res['w']}")

plt.figure(figsize=(9, 4.5))
plt.rcParams.update({'font.size': 14})
plt.plot(np.arange(res['num_iters']), res['cost_vec'])
plt.xlabel('Iteration number')
plt.ylabel('Square error C(w)')
plt.title(f"Cost vs iteration — learning rate = {res['lr']}")
plt.grid(True)
plt.show()

### Q2 (b) — extrapolate predicted expenditure 1981–2023

Because we trained on normalized variables, remember to un-normalize: feed $(\text{year}/\max\_\text{year})$ into the model and multiply the output by `max_exp`.

In [ ]:
ext_years = np.arange(1981, 2024)  # 1981..2023 inclusive
ext_X = np.ones((len(ext_years), 2))
ext_X[:, 1] = ext_years / res['max_year']
pred_norm = np.exp(-ext_X @ res['w'])
pred_exp  = pred_norm * res['max_exp']

plt.figure(figsize=(9, 4.5))
plt.scatter(res['years'], res['expenditure'], s=25, c='blue', label='real data')
plt.plot(ext_years, pred_exp, c='red', label='fitted curve')
plt.xlabel('Year')
plt.ylabel('Expenditure')
plt.title(f"Predicted education expenditure 1981–2023 (lr={res['lr']})")
plt.legend(loc='upper left')
plt.grid(True)
plt.show()

### Q2 (c) — compare $\eta \in \{0.001,\;0.03,\;0.1\}$

What to look for:
- $\eta = 0.001$ — too small; cost barely drops within the iteration budget.
- $\eta = 0.03$ — the sweet spot from (a), smooth monotone decrease.
- $\eta = 0.1$ — too large; oscillates or diverges (overshoots the minimum).

In [ ]:
N_ITERS = 200_000  # same budget for all three so the comparison is fair
results = {lr: run_gd(lr=lr, num_iters=N_ITERS) for lr in (0.001, 0.03, 0.1)}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (lr, r) in zip(axes, results.items()):
    ax.plot(np.arange(r['num_iters']), r['cost_vec'])
    ax.set_xlabel('Iteration')
    ax.set_ylabel('C(w)')
    ax.set_title(f"lr = {lr}   final = {r['final_cost']:.4f}")
    ax.grid(True)
plt.tight_layout(); plt.show()

for lr, r in results.items():
    print(f"lr = {lr:<6}  init = {r['init_cost']:.4f}  final = {r['final_cost']:.6g}  w = {r['w']}")

**Q2 takeaway.** Learning rate directly controls the speed/stability trade-off: too small → slow (barely moves in 200k iters); too large → unstable (cost explodes to `nan` or oscillates as the step jumps across the minimum). With a well-chosen $\eta$ the curve is smooth and monotone.

## Question 3 — linear model, 4th-power loss

$f(\mathbf{x},\mathbf{w}) = \mathbf{x}^{\top}\mathbf{w}$, loss $L_i = (f(\mathbf{x}_i,\mathbf{w}) - y_i)^4$, cost $C(\mathbf{w}) = \sum_i L_i$.

$$\nabla_{\mathbf{w}} C \;=\; \sum_{i=1}^{m} \nabla_{\mathbf{w}}(f(\mathbf{x}_i,\mathbf{w}) - y_i)^4$$
$$= \sum_{i=1}^{m} 4(f(\mathbf{x}_i,\mathbf{w}) - y_i)^3\, \nabla_{\mathbf{w}} f(\mathbf{x}_i,\mathbf{w}) \qquad\text{(chain rule)}$$
$$= \sum_{i=1}^{m} 4(f(\mathbf{x}_i,\mathbf{w}) - y_i)^3\, \nabla_{\mathbf{w}}(\mathbf{x}_i^{\top}\mathbf{w})$$
$$\boxed{\;\nabla_{\mathbf{w}} C = \sum_{i=1}^{m} 4\,(\mathbf{x}_i^{\top}\mathbf{w} - y_i)^3\, \mathbf{x}_i\;}$$

## Question 4 — sigmoid activation

Now $f(\mathbf{x},\mathbf{w}) = \sigma(\mathbf{x}^{\top}\mathbf{w})$ with the logistic $\sigma(a)=\dfrac{1}{1+\exp(-\beta a)}$. Q3 steps 1–3 are unchanged; only $\nabla_{\mathbf{w}} f$ changes.

**Step ①** — treat $\sigma(\mathbf{x}_i^{\top}\mathbf{w})$ as a whole and apply chain rule once more:
$$\nabla_{\mathbf{w}} C = \sum_{i=1}^{m} 4(f-y_i)^3 \,\frac{\partial \sigma(a)}{\partial a}\bigg|_{a=\mathbf{x}_i^{\top}\mathbf{w}}\, \nabla_{\mathbf{w}}(\mathbf{x}_i^{\top}\mathbf{w})$$
$$= \sum_{i=1}^{m} 4(f-y_i)^3 \,\sigma'(\mathbf{x}_i^{\top}\mathbf{w})\, \mathbf{x}_i.$$

**Step ②** — differentiate $\sigma$ once (quotient rule, then factor):
$$\sigma'(a) = \frac{\partial}{\partial a}\frac{1}{1+e^{-\beta a}} = \frac{\beta e^{-\beta a}}{(1+e^{-\beta a})^{2}} = \beta\,\sigma(a)\,(1-\sigma(a)).$$

**Put it together**:
$$\boxed{\;\nabla_{\mathbf{w}} C = \sum_{i=1}^{m} 4\,(f(\mathbf{x}_i,\mathbf{w}) - y_i)^3\, \beta\,\sigma(\mathbf{x}_i^{\top}\mathbf{w})\,(1-\sigma(\mathbf{x}_i^{\top}\mathbf{w}))\, \mathbf{x}_i\;}$$

## Question 5 — ReLU activation

Same setup, but $\sigma(a) = \max(0,a)$. Q3 skeleton unchanged; only $\sigma'$ differs.

$$\sigma'(a) = \begin{cases} 1 & a > 0 \\ 0 & a < 0 \\ \text{undefined} & a = 0 \end{cases}$$

Define the indicator $\delta(\mathbf{x}_i^{\top}\mathbf{w} > 0) = \mathbb{1}[\mathbf{x}_i^{\top}\mathbf{w} > 0]$; then (excluding the measure-zero kink at $a=0$):

$$\boxed{\;\nabla_{\mathbf{w}} C = \sum_{i=1}^{m} 4\,(f(\mathbf{x}_i,\mathbf{w}) - y_i)^3\, \mathbf{x}_i\,\delta(\mathbf{x}_i^{\top}\mathbf{w} > 0)\;}$$

**Reading the formula**: only samples whose pre-activation is positive contribute — ReLU *gates out* the rest. This is why ReLU networks can suffer from *dead neurons*: if a unit's pre-activation stays $\le 0$ across all training samples, its weights get zero gradient and never update.